In [1]:
#--- HEADER: LIBRARY IMPORTS
import json
import pandas as pd
import numpy as np
import datetime

In [2]:
#--- HEADER: LOAD JSON DATA
json_file_path = "/home/mapa8/Documents/CIDSPC/MFaD/API_call_spacex_api.json"
with open(json_file_path, 'r') as file:
    spacex_data = json.load(file)

In [3]:
# NOTE mìo, elaborando...

#--- HEADER: EXTRACT LAUNCHES LIST FROM JSON
#--- spacex_data is confirmed to be a list of launch records,
#--- so we can directly assign it without condition checking.

# Extract launches list
launches_list = spacex_data

In [4]:
#--- HEADER: CREATE MAIN DATAFRAME
#--- Converting the launches list from the JSON file into a pandas DataFrame
#--- for easier filtering, manipulation, and analysis.

data = pd.DataFrame(launches_list)

In [5]:
#--- HEADER: SUBSET COLUMNS
#--- Selecting only the columns needed for enrichment and analysis.
#--- Keeping flight_number and date_utc for temporal tracking.

# Keep only the relevant columns for data enrichment
data = data[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']]

In [6]:
#--- HEADER: FILTER OUT MULTI-CORE AND MULTI-PAYLOAD LAUNCHES
#--- Removing rows with multiple cores (Falcon Heavy with boosters) and
#--- multiple payloads (rideshare missions) to simplify analysis.
#--- Keeps only single-core, single-payload launches.

# Filter out launches with more than one core (e.g., Falcon Heavy)
data = data[data['cores'].map(len)==1]
# Filter out launches with more than one payload (rideshare missions)
data = data[data['payloads'].map(len)==1]

In [7]:
#--- HEADER: EXTRACT SINGLE VALUES FROM LISTS
#--- Since payloads and cores are now lists of size 1, extract the
#--- single value and replace the list with the value itself.

# Extract the single core ID from the list and convert to string
data['cores'] = data['cores'].map(lambda x: x[0])

# Extract the single payload ID from the list and convert to string
data['payloads'] = data['payloads'].map(lambda x: x[0])

# Extract the single core ID from the list and convert to string
# lambda x: x[0] takes each value (which is a list) and returns the first element at index 0
# Example: ['5e9e289df35918066d3b2623'] becomes '5e9e289df35918066d3b2623'

In [8]:
# NOTE mìo...

#--- HEADER: EXTRACT CORE ID FROM DICTIONARY
#--- The 'cores' column contains dictionaries with a 'core' key.
#--- This lambda extracts the value of the 'core' key from each dictionary.

# Extract the single core ID from the dictionary and convert to string
# lambda x: x['core'] takes each value (which is a dictionary) and returns the value of the 'core' key
# Example: {'core': '5e9e289df35918033d3b2623', 'flight': 1, ...} becomes '5e9e289df35918033d3b2623'

data['cores'] = data['cores'].map(lambda x: x['core'] if isinstance(x, dict) else x)

# NOTE èxito!

In [9]:
#--- HEADER: DATE CONVERSION AND FILTERING
#--- Converting date_utc from string to datetime, extracting just the date,
#--- and filtering to include only launches up to November 13, 2020.

# Convert date_utc to datetime and extract only the date portion
data['date'] = pd.to_datetime(data['date_utc']).dt.date
# Keep only launches up to November 13, 2020 (course dataset timeframe)
data = data[data['date'] <= datetime.date(2020, 11, 13)]

In [10]:
#--- HEADER: INITIALIZE GLOBAL LISTS
#--- Creating empty lists that will store extracted data for each launch.
#--- Each list will be populated by the extraction functions and then
#--- combined into the final DataFrame.

BoosterVersion = []   # Rocket name (e.g., Falcon 9)
Longitude = []        # Launch site longitude
Latitude = []         # Launch site latitude
LaunchSite = []       # Launch site name (e.g., CCAFS SLC-40)
PayloadMass = []      # Payload mass in kg
Orbit = []            # Target orbit (e.g., LEO, GTO)
Block = []            # Booster version block number
ReusedCount = []      # Number of times core has been reused
Serial = []           # Core serial number (e.g., B1049)
Outcome = []          # Landing outcome (success + landing type)
Flights = []          # Flight number for this core
GridFins = []         # Whether grid fins were used (True/False)
Reused = []           # Whether core was reused (True/False)
Legs = []             # Whether landing legs were used (True/False)
LandingPad = []       # Landing pad used (e.g., JRTI-1)

In [11]:
#--- HEADER: BUILD LOOKUP DICTIONARIES
#--- Creating empty dictionaries that will store rocket, launchpad, payload,
#--- and core data from the JSON file for fast ID-based lookups.
#--- Each dictionary will map an ID (string) to its full data (dictionary).

rockets_data = {}
launchpads_data = {}
payloads_data = {}
cores_data = {}

In [12]:
#--- HEADER: BUILD LOOKUP DICTIONARIES FROM DATAFRAME
#--- Extracting unique rocket, launchpad, payload, and core IDs
#--- from the data DataFrame to create lookup dictionaries.
#--- Each dictionary maps an ID string to a minimal dictionary.

"""
# NOTE mìo, diccionarios ya definidos
# rockets_data = {}
# launchpads_data = {}
# payloads_data = {}
# cores_data = {}
"""

# Build dictionaries from the string IDs in the DataFrame
for idx, row in data.iterrows():
    # Rocket - string ID
    rocket = row['rocket']
    if rocket:
        rockets_data.setdefault(rocket, {'id': rocket})
    
    # Launchpad - string ID
    launchpad = row['launchpad']
    if launchpad:
        launchpads_data.setdefault(launchpad, {'id': launchpad})
    
    # Payloads - string ID
    payload = row['payloads']
    if payload:
        payloads_data.setdefault(payload, {'id': payload})
    
    # Cores - string ID
    core = row['cores']
    if core:
        cores_data.setdefault(core, {'id': core})

In [13]:
#
# ! DISMISSED - DATASETS MISSING

# 	api.spacexdata.com/v4/
# 	[https://gateway.pipeworx.io/spacex/]
# 		rockets 		    [yes]	->	for_getBoosterVersion_rockets.json
# 		launchpads		    [no]
# 		payloads		    [no]
# 		cores			    [no]
# 		* launches/past		[yes]	->	from_spacex_url_response.json
